# Experiment A: Differential Error Detection by Claim Type

## CCL Prediction
Factual errors are more reliably detected than interpretive errors, which are more reliably detected than missing perspectives. This justifies the CCL's claim-type annotation system.

## Data
**FRANK benchmark** (Pagnoni et al., NAACL 2021). 4,942 sentences from AI-generated summaries, each annotated by 3 trained annotators across 7 error types.

## CCL Mapping
- **FACTUAL** (■): EntE + OutE + GramE
- **SOURCE** (♦): CircE
- **INTERPRETIVE** (▲): RelE + LinkE + CorefE
- **Missing perspectives** (•): no FRANK equivalent

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../..')

import matplotlib.pyplot as plt
from experiments.shared.plotting import setup_style, grouped_bar_chart, ccl_palette
setup_style()

## 1. Load FRANK Data

In [ ]:
from experiments.shared.data_acquisition import load_frank, frank_to_sentence_df

raw = load_frank()
sentence_df = frank_to_sentence_df(raw)
print(f"{len(sentence_df)} sentences from {len(raw)} summaries")
sentence_df.head()

## 2. Build CCL Category Matrix

In [ ]:
from experiments.exp_a_error_detection.analysis import build_sentence_category_matrix

matrix = build_sentence_category_matrix(sentence_df)
print(f"{len(matrix)} rows (sentences × categories)")
matrix.head(10)

## 3. Per-Category Statistics

In [ ]:
from experiments.exp_a_error_detection.analysis import compute_category_stats

stats_df = compute_category_stats(matrix)
stats_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

agreement_data = dict(zip(stats_df['ccl_category'], stats_df['majority_agreement']))
grouped_bar_chart(agreement_data, 'Majority Agreement Rate', 'Proportion (≥2/3 agree)', ax=axes[0])

kappa_data = dict(zip(stats_df['ccl_category'], stats_df['fleiss_kappa']))
grouped_bar_chart(kappa_data, "Fleiss' κ by Category", "Fleiss' κ", ax=axes[1])

fig.suptitle('Experiment A: Error Detection Reliability by CCL Category', fontweight='bold')
fig.tight_layout()
plt.show()

## 4. Fisher's Exact Test (Factual vs. Interpretive)

In [ ]:
from experiments.exp_a_error_detection.analysis import fisher_test_factual_vs_interpretive

fisher = fisher_test_factual_vs_interpretive(matrix)
print(f"Odds ratio: {fisher['odds_ratio']:.2f}")
print(f"p-value:    {fisher['p_value']:.2e}")

## 5. Interpretation

When a factual error exists in AI-generated text, annotators agree on its presence at a much higher rate than for interpretive errors. Even trained NLP annotators show a substantial detection reliability gap between factual and interpretive errors.

**Implication for the CCL**: Without explicit scaffolding, interpretive claims pass unquestioned. This validates the CCL's claim-type annotation system: the Stage 2 checklist and Stage 4 adversarial challenge are designed to address precisely this asymmetry.

**Limitation**: FRANK annotators are trained crowdworkers, not students. The mapping from FRANK's summarisation error taxonomy to CCL's claim-type categories is analogical, not exact.